# Figure 3.7: Two-dimensional convolution

*The Discrete Algebra of the Fourier Transform*, Gabriel Peyré.

Circular Gaussian smoothing of a noisy surface. The discrete kernel has unit sum; its center is shifted to index $(0,0)$ before the FFT.

[Read this figure in the book](https://mathematical-tours.github.io/daft-sources/DAFT-EN.pdf#page=63).

## Prepare the example

Run the cells in order in Jupyter or Google Colab. The first cell installs missing Python packages and downloads the figure sources and input data once. Subsequent runs use the cached download. Python 3.10 or later is required.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import io
import os
import subprocess
import sys
import urllib.request
import zipfile

packages = {"numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib",
            "PIL": "Pillow", "sympy": "sympy"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

archive_url = os.environ.get("DAFT_FIGURE_ARCHIVE", 'https://mathematical-tours.github.io/daft-sources/downloads/daft-figures-2e843bbe89f0.zip')
archive_sha = '2e843bbe89f0e274849b6896eb16bd9f7686521a8ca98f633240bfc06ea2d5b1'
workspace = Path(os.environ.get("DAFT_FIGURE_WORKDIR", "daft-figure-files")) / archive_sha[:12]
cache = workspace / "sources.zip"
workspace.mkdir(parents=True, exist_ok=True)
data = cache.read_bytes() if cache.exists() else urllib.request.urlopen(archive_url, timeout=60).read()
if hashlib.sha256(data).hexdigest() != archive_sha:
    raise ValueError("The figure-source download is incomplete or has changed. Download it again.")
cache.write_bytes(data)
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    archive.extractall(workspace)
ROOT = (workspace / "daft-figures").resolve()
__file__ = str(ROOT / 'figures-code/chapter-discrete-fourier-transform/two-dimensional-convolution/generate.py')
print("Figure sources ready.")


## Generate the figure

This is the plotting code used by the book. Change its parameters and rerun this cell, then the display cell below, to explore the example.

In [ ]:
from pathlib import Path
import sys
ROOT = Path(__file__).resolve().parents[3]
sys.path.insert(0, str(ROOT / "tools"))
sys.path.insert(0, str(ROOT / "listings"))
from figure_style import *
n=48;x=np.linspace(-3,3,n,endpoint=False);X,Y=np.meshgrid(x,x)
f=3*(1-X)**2*np.exp(-X**2-(Y+1)**2)-10*(X/5-X**3-Y**5)*np.exp(-X**2-Y**2)-np.exp(-(X+1)**2-Y**2)/3
f+=.55*np.random.default_rng(20260908).standard_normal((n,n))
k=np.arange(-n//2,n//2);K,L=np.meshgrid(k,k);g=np.exp(-(K*K+L*L)/(2*1.8**2));g/=g.sum()
smoothed=np.fft.ifft2(np.fft.fft2(f)*np.fft.fft2(np.fft.ifftshift(g))).real
assert np.isclose(f.sum(),smoothed.sum())
fig=plt.figure(figsize=(11.5,3.4))
for panel,(Z,title) in enumerate([(f,'Noisy surface $f$'),(g,'Unit-sum Gaussian $g$'),(smoothed,r'Circular convolution $f*g$')]):
    ax=fig.add_subplot(1,3,panel+1,projection='3d')
    ax.plot_surface(X,Y,Z,cmap='viridis',rstride=1,cstride=1,linewidth=0,antialiased=True,rasterized=True)
    ax.set(xlabel='$x$',ylabel='$y$',title=title,xticks=[-3,0,3],yticks=[-3,0,3])
    if panel!=1:ax.set_zlim(-7,9)
    ax.view_init(28,-58);ax.set_box_aspect((1,1,.68));ax.tick_params(pad=0,labelsize=8)
    for axis in [ax.xaxis,ax.yaxis,ax.zaxis]:axis.pane.fill=False
save(fig,'chapter-discrete-fourier-transform','two-dimensional-convolution')


## Display and save

In [ ]:
from IPython.display import Image, FileLink, display

display(Image(filename=str(ROOT / 'build/figure-previews/chapter-discrete-fourier-transform/two-dimensional-convolution.png')))
display(FileLink(os.path.relpath(ROOT / 'figures/chapter-discrete-fourier-transform/two-dimensional-convolution/figure.pdf', Path.cwd()),
                 result_html_prefix="Download the vector PDF: "))
